## Ноутбук с подготовкой и очисткой данных

1. Импорт библиотек и конфигурация проекта

In [1]:
# Создадим словарь конфигураций.

CONFIG = {
    # Константы
    "DEV_MODE": False,
    "DEV_SAMPLE_SIZE": 100000,
    "RANDOM_STATE": 42,
    # Целевая переменная 
    "TARGET": "Цена",
}

In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pyarrow

2. Загрузка и первичный осмотр данных

In [ ]:
dates_only = pd.read_csv('../data/raw/raw_dataset.csv', usecols=['Дата размещения объявления'])
print("Самая поздняя дата в файле:", dates_only['Дата размещения объявления'].max())

Заметим, что самая поздняя дата объявления - 2025-07-14. Так как файл слишком большой (5гб), отберём только объявления, размещенные с 2025-01-14 по 2025-07-14: это позволит не только облегчить вычисления, но и сделает будущую модель лучше, ведь она будет обучена на относительно "свежих" данных (учитываем инфляцию и актуальность цен).

In [ ]:
date = 'Дата размещения объявления'
chunks = []
for chunk in pd.read_csv('../data/raw/raw_dataset.csv', chunksize = 100000, low_memory=False):
    filtered = chunk[chunk[date].between('2025-01-14','2025-07-14')]
    chunks.append(filtered)

df = pd.concat(chunks, ignore_index = True)
df.shape

(585855, 58)

585855 строк - оптимальное значенение для обучения модели. Больше брать смысла нет, так как качество модели растет логарифмически по отношению к объему данных. Для начала проверим, есть ли в нашем датасете информация о спецтехнике.

In [ ]:
trucks_count = df['Тип техники'].notna().sum()
print(f"Найдено коммерческой техники/спецтехники: {trucks_count} шт.")

Найдено коммерческой техники/спецтехники: 0 шт.


Отлично! Никаких грузовиков, тягачей и кранов в нашем полном датасете нет - можно спокойно работать с нашим последующим сэмплом в 100к, не боясь что мы удалим важные столбцы для нелегковых автомобилей.
Все эксперименты будем проводить на DEV_MODE = True, чтобы работать с 100тыс. строк. В конце работы в CONFIG поменяем значение на False -> финальный запуск на всем объеме (585к строк)

In [ ]:
if CONFIG['DEV_MODE']:
    df = df.sample(n=100000, random_state=CONFIG['RANDOM_STATE'])
    print("Режим разработки (100к строк). Всё будет летать!")
else:
    df = df.copy()
    print("Финальный режим (585к строк). Обучаем итоговую модель.")

Финальный режим (585к строк). Обучаем итоговую модель.


In [ ]:
df.info

<bound method DataFrame.info of              Название машины     Год  \
0       Aston Martin Vantage  2018.0   
1           Aston Martin DB9  2005.0   
2          Aston Martin DB11  2017.0   
3           Aston Martin DB9  2013.0   
4           Aston Martin DBS  2019.0   
...                      ...     ...   
585850            Volvo XC90  2006.0   
585851             Volvo C30  2008.0   
585852             Volvo V90  2019.0   
585853             Volvo V60  2018.0   
585854             Volvo S80  2002.0   

                                                   Ссылка  \
0       https://auto.drom.ru/himki/aston_martin/vantag...   
1       https://auto.drom.ru/krasnodar/aston_martin/db...   
2       https://auto.drom.ru/moscow/aston_martin/db11/...   
3       https://auto.drom.ru/moscow/aston_martin/db9/1...   
4       https://auto.drom.ru/moscow/aston_martin/dbs/5...   
...                                                   ...   
585850  https://auto.drom.ru/moscow/volvo/xc90/5051113...   

In [ ]:
df.head(5)

,Название машины,Год,Ссылка,Дата размещения объявления,Цена,Кол-во просмотров,Скрыто,Объем двигателя,Тип двигателя,Мощность,...,Объем ковша,Длина стрелы,Грузоподъемность стрелы,Высота вышки,Состояние,Страна производства,Высота подъема,Ошибка_ст,Ошибка_знач,Пропуски в данных
0,Aston Martin Vantage,2018.0,https://auto.drom.ru/himki/aston_martin/vantag...,2025-04-03,11834000.0,1899.0,0.0,4.0,бензин,510.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"20,","открытый,","17,"
1,Aston Martin DB9,2005.0,https://auto.drom.ru/krasnodar/aston_martin/db...,2025-04-18,4999000.0,1230.0,0.0,5.9,бензин,456.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"9,","510.0,","17,"
2,Aston Martin DB11,2017.0,https://auto.drom.ru/moscow/aston_martin/db11/...,2025-05-16,13900000.0,1317.0,0.0,5.2,бензин,608.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Aston Martin DB9,2013.0,https://auto.drom.ru/moscow/aston_martin/db9/1...,2025-03-29,8500000.0,14571.0,0.0,5.9,бензин,510.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aston Martin DBS,2019.0,https://auto.drom.ru/moscow/aston_martin/dbs/5...,2025-03-31,24300000.0,11888.0,0.0,5.2,бензин,715.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
(df.isnull().mean() * 100).round(2)

Название машины                 0.00
Год                             0.00
Ссылка                          0.00
Дата размещения объявления      0.00
Цена                            0.00
Кол-во просмотров               0.00
Скрыто                          0.00
Объем двигателя                 0.01
Тип двигателя                   0.00
Мощность                        0.01
Коробка передач                 0.00
Привод                          0.00
Пробег                          0.80
Руль                            0.04
Поколение                       0.01
Рестайлинг                      0.01
Цвет                            0.38
Комплектация                    0.22
Владелец                        0.00
Особые отметки                 93.42
Тип кузова                      0.40
VIN                            99.11
Проверено                     100.00
Номер кузова                   99.99
Метка                           0.00
Город                           0.00
Регион                          0.00
М

Заметим, что такие данные как высота подъема, объем ковша, высота седла, тип кабины и др. на 100% отсутствуют. Это связано с тем, что в данном датасете только легковые автомобили. Сможем смело удалять эти столбцы

3. Очистка данных

In [ ]:
# Перед удалением создадим отдельный столбец, тк особые отметки - очень важный признак
df['Есть особые отметки'] = df['Особые отметки'].notna().astype(int)
# Задаем порог: если пропусков больше, чем 50% - смело удаляем столбец
threshold = len(df) * 0.5  
df_cleaned = df.dropna(thresh=threshold, axis=1).copy()
print('Было колонок: ', df.shape[1])
print('Стало колонок: ', df_cleaned.shape[1])

Было колонок:  59
Стало колонок:  27


In [ ]:
# Удаляем строчки, у которых отсутствует пробег - всего 0.77 от датасета, это ни на что не повлияет
df_cleaned = df_cleaned.dropna(subset=['Пробег'])

# Избавляемся от дубликатов
df_cleaned.drop_duplicates(inplace=True)

In [ ]:
# Убираем ненужны столбцы
cols_to_drop = ['Дата размещения объявления', 'Кол-во просмотров', 'Скрыто', 'Ссылка', 'Владелец', 'Пропуски в данных']
df_cleaned.drop(columns=cols_to_drop, inplace=True)

In [ ]:
cols_to_unknown = ['Цвет', 'Комплектация']
for col in cols_to_unknown:
    df_cleaned[col] = df_cleaned[col].fillna('Unknown')

In [ ]:
# Переименовываем "Метку" в понятную "Марку"
df_cleaned.rename(columns={'Метка': 'Марка'}, inplace=True)

4. Разделение на train и test

Критический шаг. Делаем это для избежания утечки данных

In [ ]:
X = df_cleaned.drop(columns=[CONFIG['TARGET']])
y = df_cleaned[CONFIG['TARGET']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=CONFIG['RANDOM_STATE'])

5. Контекстная очистка

In [ ]:
# Заполним медианой столбцы с числовыми признаками

num_cols = ['Мощность', 'Объем двигателя']
for col in num_cols:
    global_median = X_train[col].median()
    group_medians = X_train.groupby('Название машины')[col].median()
    X_train[col] = X_train[col].fillna(
        X_train['Название машины'].map(group_medians)
        ).fillna(global_median)
    X_test[col] = X_test[col].fillna(
        X_test['Название машины'].map(group_medians)
    ).fillna(global_median)

In [ ]:
# Заполним модой столбцы с категориальными признаками

cat_cols = ['Привод', 'Руль', 'Тип кузова', 'Тип двигателя', 'Владельцы', 'Коробка передач']
for col in cat_cols:
    global_mode = X_train[col].mode()[0]
    group_modes = X_train.groupby('Название машины')[col].apply(
        lambda x: x.mode().get(0, global_mode)
    )
    X_train[col] = X_train[col].fillna(
        X_train['Название машины'].map(group_modes)
    ).fillna(global_mode)
    X_test[col] = X_test[col].fillna(
        X_test['Название машины'].map(group_modes)
    ).fillna(global_mode)

In [ ]:
# Особенный случай - поколение и рестайлинг зависят и от модели, и от года выпуска
special_cols = ['Поколение', 'Рестайлинг']
for col in special_cols:
    global_mode = X_train[col].mode()[0]
    group_modes = X_train.groupby(['Название машины', 'Год'])[col].apply(
        lambda x: x.mode().get(0, global_mode)
    ).rename(f'mode_{col}')

    X_train[col] = X_train[col].fillna(
        X_train.join(group_modes, on=['Название машины', 'Год'])[f'mode_{col}']
    ).fillna(global_mode)

    X_test[col] = X_test[col].fillna(
        X_test.join(group_modes, on=['Название машины', 'Год'])[f'mode_{col}']
    ).fillna(global_mode)

In [ ]:
# Проверим
display(X_test.isna().sum())
X_train.isna().sum()

Название машины        0
Год                    0
Объем двигателя        0
Тип двигателя          0
Мощность               0
Коробка передач        0
Привод                 0
Пробег                 0
Руль                   0
Поколение              0
Рестайлинг             0
Цвет                   0
Комплектация           0
Тип кузова             0
Марка                  0
Город                  0
Регион                 0
Макро-регион           0
Владельцы              0
Есть особые отметки    0
dtype: int64

Название машины        0
Год                    0
Объем двигателя        0
Тип двигателя          0
Мощность               0
Коробка передач        0
Привод                 0
Пробег                 0
Руль                   0
Поколение              0
Рестайлинг             0
Цвет                   0
Комплектация           0
Тип кузова             0
Марка                  0
Город                  0
Регион                 0
Макро-регион           0
Владельцы              0
Есть особые отметки    0
dtype: int64

6. Проверка выбросов

In [ ]:
X_train.columns

Index(['Название машины', 'Год', 'Объем двигателя', 'Тип двигателя',
       'Мощность', 'Коробка передач', 'Привод', 'Пробег', 'Руль', 'Поколение',
       'Рестайлинг', 'Цвет', 'Комплектация', 'Тип кузова', 'Марка', 'Город',
       'Регион', 'Макро-регион', 'Владельцы', 'Есть особые отметки'],
      dtype='str')

Посмотрим на год

In [ ]:
X_train['Год'].value_counts().sort_index().head(30)

Год
1941.0       1
1949.0       1
1950.0       1
1953.0       1
1959.0       1
1962.0       1
1965.0       1
1967.0       1
1970.0       1
1971.0       1
1972.0       5
1973.0       8
1974.0      22
1975.0      34
1976.0      27
1977.0      45
1978.0      36
1979.0      56
1980.0      84
1981.0     123
1982.0     214
1983.0     318
1984.0     519
1985.0     590
1986.0     653
1987.0     844
1988.0    1311
1989.0    1545
1990.0    2171
1991.0    2790
Name: count, dtype: int64

Избавимся от ретро-автомобилей. Они будут ломать логику модели, ведь для обычных авто действует правило "чем старше, тем он дешевле" (цена падает с возрастом). Для ретро-машин "чем старше и раритетнее авто, тем он дороже".

In [ ]:
df = df[df['Год'] >= 1990]

Посмотрим на объем двигателя

In [ ]:
X_train['Объем двигателя'].value_counts().sort_index().head(20)

Объем двигателя
0.000000     928
0.500000       1
0.600000      62
0.700000       2
0.700000    3909
0.800000     182
0.820000       1
0.900000       1
1.000000    6888
1.066667       1
1.071429       1
1.075000       1
1.077778       1
1.090909       1
1.100000     483
1.133333       3
1.150000       2
1.200000    6529
1.214286       1
1.214815       1
Name: count, dtype: int64

In [ ]:
X_train.sort_values(by='Объем двигателя', ascending=True).head(10)

,Название машины,Год,Объем двигателя,Тип двигателя,Мощность,Коробка передач,Привод,Пробег,Руль,Поколение,Рестайлинг,Цвет,Комплектация,Тип кузова,Марка,Город,Регион,Макро-регион,Владельцы,Есть особые отметки
380631,Nissan Leaf,2012.0,0.0,электро,109.00,редуктор,передний,95000.0,правый,1.0,0.0,белый,G,хэтчбек 5 дв.,nissan,Можайск,Московская область,ЦФО,1,1
486037,Toyota bZ4X,2022.0,0.0,электро,204.00,редуктор,передний,36000.0,левый,1.0,0.0,черный,71.4 kWh Comfort-Paket,джип/suv 5 дв.,toyota,Симферополь,Республика Крым,ЮФО,1,0
140134,Kia EV5,2024.0,0.0,электро,218.00,редуктор,передний,0.0,левый,1.0,0.0,серый,64.2 kWh 530 Air//64.2 kWh 530 Land//64.2 kWh ...,джип/suv 5 дв.,kia,Минск,Минск,Беларусь,4,0
408632,Porsche Taycan,2020.0,0.0,электро,625.00,редуктор,4WD,88000.0,левый,1.0,0.0,белый,93.4 kWh Turbo,седан,porsche,Москва,Москва,Москва,1,0
17174,BMW i3,2015.0,0.0,электро,170.00,редуктор,задний,53000.0,левый,1.0,0.0,серый,i3 60 Ah,хэтчбек 5 дв.,bmw,Ижевск,Удмуртская Республика,ПФО,2,0
391282,Nissan Leaf,2014.0,0.0,электро,109.00,редуктор,передний,71500.0,правый,1.0,0.0,серый,X Aero Style,хэтчбек 5 дв.,nissan,Ангарск,Иркутская область,СФО,1,0
408629,Porsche Taycan,2021.0,0.0,электро,564.25,редуктор,4WD,31000.0,левый,1.0,0.0,красный,79.2 kWh 4S//93.4 kWh GTS//93.4 kWh Turbo//93....,седан,porsche,Минск,Минск,Беларусь,1,0
140137,Kia EV5,2024.0,0.0,электро,218.00,редуктор,передний,0.0,левый,1.0,0.0,серый,64.2 kWh 530 Air//64.2 kWh 530 Land//64.2 kWh ...,джип/suv 5 дв.,kia,Санкт-Петербург,Санкт-Петербург,Санкт-Петербург,4,0
578081,Volkswagen ID.4,2023.0,0.0,электро,313.00,редуктор,4WD,0.0,левый,1.0,0.0,серый,84.8 kWh Prime,джип/suv 5 дв.,volkswagen,Тюмень,Тюменская область,УрФО,1,0
383573,Nissan Leaf,2015.0,0.0,электро,109.00,редуктор,передний,196000.0,правый,1.0,0.0,белый,24kWh S side/curtain airbag system less,хэтчбек 5 дв.,nissan,Челябинск,Челябинская область,УрФО,1,0


Заметим, что все автомобили, у которых объем двигателя равен нуля - это электромобили. Ценные данные, их удалять нельзя.

In [ ]:
X_train['Объем двигателя'].value_counts().sort_index(ascending=False).head(30)

Объем двигателя
34.000000      1
15.000000      1
8.200000       1
8.000000       1
7.400000       1
7.300000       3
7.000000       2
6.800000       6
6.700000      18
6.600000      10
6.500000       8
6.400000      28
6.300000       5
6.200000     267
6.100000       6
6.000000     105
5.900000       6
5.800000      24
5.700000     745
5.666667       1
5.600000     522
5.500000     586
5.400000      85
5.300000      97
5.250000       2
5.242857       1
5.228571       1
5.200000      60
5.090909       1
5.080000       1
Name: count, dtype: int64

После значения 6.8 данные резко обрываются, остаются единичные машины. Значение 0.50 - скорее всего единичный выброс. Избавимся от этого.

In [ ]:
X_train = X_train[(X_train['Объем двигателя'] == 0) | ((X_train['Объем двигателя'] >= 0.6) & (X_train['Объем двигателя'] <= 6.8))]
X_test = X_test[(X_test['Объем двигателя'] == 0) | ((X_test['Объем двигателя'] >= 0.6) & (X_test['Объем двигателя'] <= 6.8))]

Посмотрим на тип двигателя.

In [ ]:
X_train['Тип двигателя'].value_counts().sort_index(ascending=False).head(10)

Тип двигателя
электро          928
дизель         27878
газ/бензин         2
газ               30
бензин        436074
Name: count, dtype: int64

Газ/бензин - это реальный тип двигателя. Это не ошибка. Убирать не будем

Посмотрим на мощность.

In [ ]:
X_train['Мощность'].value_counts().sort_index(ascending=False).head(10)

Мощность
1952.0    1
1560.0    1
800.0     4
780.0     6
760.0     1
740.0     1
725.0     1
720.0     3
715.0     2
700.0     2
Name: count, dtype: int64

Удалим выбросы в виде значений 1952 и 1560

In [ ]:
X_train = X_train[X_train['Мощность'] <= 800]
X_test = X_test[X_test['Мощность'] <= 800]

Посмотрим на простые признаки

In [ ]:
# Быстрая проверка простых категориальных признаков на неявные дубликаты и аномалии
check_cols = ['Коробка передач', 'Привод', 'Тип кузова', 'Цвет', 'Рестайлинг', 'Поколение', 'Комплектация', 'Метка', 'Город', 'Макро-регион', 'Регион']

for col in check_cols:
    print(f"\tРаспределение для столбца '{col}'")
    print(df[col].value_counts())
    print("\n" + "-"*40 + "\n")

	Распределение для столбца 'Коробка передач'
Коробка передач
АКПП        248962
МКПП        224195
CVT          77489
РКПП         23884
редуктор      3108
Name: count, dtype: int64

----------------------------------------

	Распределение для столбца 'Привод'
Привод
передний                      377253
4WD                           147501
задний                         52767
двигатель посередине (MID)       114
Name: count, dtype: int64

----------------------------------------

	Распределение для столбца 'Тип кузова'
Тип кузова
седан                                                                                                                                                                   236073
джип/suv 5 дв.                                                                                                                                                          120063
хэтчбек 5 дв.                                                                                                     

Данные чистые.

Посмотрим на владельцев.

In [ ]:
X_train['Владельцы'].value_counts()

Владельцы
4 и более    241714
1             97696
3             62705
2             61498
1.0             930
2.0             218
3.0             165
Name: count, dtype: int64

Заметим, что данные смешались. Попробуем привести всё к единому формату

In [ ]:
owners_mapping = {
    '4 и более': 4, # Стандартное упрощение для моделей. Оно дает ей понять направление тренда (что владельцев много), не усложняя вычисления.
    '1.0': 1,
    '2.0': 2,
    '3.0': 3,
    1.0: 1,
    2.0: 2,
    3.0: 3,
    '1': 1,
    '2': 2,
    '3': 3
}

# Применяем замену к столбцу
X_train['Владельцы'] = X_train['Владельцы'].replace(owners_mapping)
X_test['Владельцы'] = X_test['Владельцы'].replace(owners_mapping)

Посмотрим на руль.

In [ ]:
X_train['Руль'].value_counts()

Руль
левый     327247
правый    137665
Name: count, dtype: int64

In [ ]:
# Удаляем этот выброс.
X_train = X_train[X_train['Руль'] != 'правый, левый']
X_test = X_test[X_test['Руль'] != 'правый, левый']

Посмотрим на пробег.

In [ ]:
X_train['Пробег'].value_counts().sort_index(ascending=False).head(10)

Очевидно, что почти 300 машин не могут иметь ровно 999999 км пробег. Заменим на медиану по году выпуска.

In [ ]:
df.loc[df['Пробег'] >= 999000, 'Пробег'] = np.nan
df['Пробег'] = df['Пробег'].fillna(df.groupby('Год')['Пробег'].transform('median'))

7. Объединение и импорт

In [ ]:
train_full = pd.concat([X_train, y_train], axis=1)
test_full = pd.concat([X_test, y_test], axis=1)

train_full.to_parquet('../data/processed/train_cleaned.parquet')
test_full.to_parquet('../data/processed/test_cleaned.parquet')